# dgx-spark-hijinks: sm_120 probe matrix (Colab Pro, RTX PRO 6000 / G4)

**Version: JACKAL** - version names are animals, alphabetical, bumped on every push
(Axolotl, Badger, Capybara, Dingo = verified CUDA-13 apt bootstrap). If the top of
your copy says an earlier animal, re-open from GitHub.

**Purpose:** re-run the campaign's weight-free FlashInfer FA2 NVFP4/VO-split probes on
**sm_120** (RTX PRO 6000 class) to test whether every dispatcher finding measured on
GB10 (**sm_121**) is CC 12.x **family-wide**. No models, no weights — megabytes of
synthetic tensors + JIT kernel compiles. ~30–45 min total, most of it the first JIT.

**Expected results (GB10 reference, branch `fb7d62ea`):**

| probe | GB10 result |
|---|---|
| A1 vo-split d512 FP4-KV, batch 4 | PASS cosine ≥ 0.9999985 |
| A2 vo-split, qo_len=1 (decode shape) | PASS ≥ 0.9999984 |
| A3 vo-split, signed E2M1 | PASS ≥ 0.9999983 |
| A4 d128 linear-SF control (decode+prefill) | PASS ≥ 0.9999994 |
| geometry e4b (GQA group 4) bf16 vo-split | PASS 0.9999979 |
| geometry 31b (32q/16kv) bf16 vo-split | PASS 0.9999980 |
| fp8-KV (512,256) | **RED**: trait guard `Invalid configuration ... NUM_MMA_KV=1 NUM_WARPS_Q=4` |
| bf16 d512 symmetric (no split) | **RED**: trait guard (the D=512 wall) |

Matching greens AND matching reds = family-wide. Any divergence is a per-arch finding —
either way it goes in the ledger and strengthens the upstream issues.

In [ ]:
# Cell 1 — GPU + environment check. Expect compute capability (12, 0).
NOTEBOOK_VERSION = 'JACKAL'  # KEEP IN SYNC with the banner in the title cell
import torch, subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout)
cap = torch.cuda.get_device_capability(0)
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| capability', cap)
assert cap[0] == 12, f'Need a CC 12.x GPU (G4 / RTX PRO 6000 runtime); got {cap}'
if cap != (12, 0):
    print(f'NOTE: capability {cap} != (12,0) — results still useful, label them accordingly')

In [ ]:
# Cell 1b - CUDA >= 12.9 bootstrap, step 1 (FlashInfer refuses SM 12.x below 12.9;
# stock Colab ships 12.8). Run once; if it installs anything, Runtime -> Restart
# session, then continue from the NEXT cell. Skips itself if already satisfied.
import torch
cu = tuple(map(int, torch.version.cuda.split('.')[:2]))
if cu >= (12, 9):
    print(f'CUDA {torch.version.cuda} OK - no bootstrap needed, continue to next cell')
else:
    print(f'CUDA {torch.version.cuda} < 12.9 - installing cu130 torch + nvcc 13 (~3-5 min)')
    import subprocess
    subprocess.run('pip install -q --upgrade --index-url https://download.pytorch.org/whl/cu130 torch torchvision', shell=True)
    # NOTE: pip 'nvidia-cuda-nvcc-cu13' is ptxas-only (no compiler driver) - not installed
    print('*** NOW: Runtime -> Restart session, then run from the NEXT cell (1c) ***')


In [ ]:
# Cell 1c - post-restart: full nvcc 13 via NVIDIA apt repo (~1-2 min).
# (The pip cu13 'nvcc' wheel ships only ptxas + headers - no compiler driver -
# so a pip-assembled CUDA_HOME is impossible; verified dead end 2026-06-11.)
import subprocess, os, torch
assert tuple(map(int, torch.version.cuda.split('.')[:2])) >= (12, 9), 'rerun cell 1b + restart first'
have = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
if have.returncode == 0 and 'release 13' in have.stdout:
    print('nvcc 13 already present')
else:
    ub = '2404' if '24.04' in open('/etc/os-release').read() else '2204'
    for c in [
        f'wget -q https://developer.download.nvidia.com/compute/cuda/repos/ubuntu{ub}/x86_64/cuda-keyring_1.1-1_all.deb -O /tmp/ck.deb',
        'dpkg -i /tmp/ck.deb',
        'apt-get -qq update',
        'apt-get -qq install -y cuda-toolkit-13-0',  # wholesale: ~3GB/8min, but DETERMINISTIC - the minimal set cost five user round-trips (nvcc, nvrtc, cusparse, cublas...)
    ]:
        print('+', c)
        r = subprocess.run(c, shell=True, capture_output=True, text=True)
        if r.returncode: print(r.stderr[-500:])
os.environ['CUDA_HOME'] = '/usr/local/cuda-13.0'
os.environ['PATH'] = '/usr/local/cuda-13.0/bin:' + os.environ['PATH']
r = subprocess.run(['/usr/local/cuda-13.0/bin/nvcc', '--version'], capture_output=True, text=True)
print('nvcc:', r.stdout.splitlines()[-1] if r.returncode == 0 else r.stderr[-300:])


In [ ]:
# Cell 2 — (optional but recommended) Drive for results + JIT-cache persistence.
# CAMPAIGN RULE: never reuse a Drive JIT cache across V-SF-mode changes
# (results/jit_cache_mode_unsoundness_analysis_20260611.md). This notebook uses
# linear-SF probes only, so one cache dir is fine.
import os
RESULTS_DIR = '/content/results_colab_sm120'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = '/content/drive/MyDrive/dgx-spark-hijinks/results_colab_sm120'
    os.environ['FLASHINFER_JIT_DIR'] = '/content/drive/MyDrive/dgx-spark-hijinks/fi_jit_sm120_linear'
except Exception as e:
    print('No Drive (fine, results are ephemeral):', e)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('results ->', RESULTS_DIR)

In [ ]:
# Cell 3 — clone branches; install FlashInfer (editable if possible, else source-tree mode).
import subprocess, sys, os
def sh(cmd, check=True):
    print('+', cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.stdout: print(r.stdout[-2000:])
    if r.stderr: print(r.stderr[-2000:])
    if check and r.returncode != 0:
        raise RuntimeError(f'failed ({r.returncode}): {cmd}')
    return r

if not os.path.isdir('/content/flashinfer'):
    sh('git clone --recursive https://github.com/jethac/flashinfer -b spark/hijinks-022-fa2-d512 /content/flashinfer')
sh('cd /content/flashinfer && git checkout fb7d62ea && git submodule update --init --recursive')
if not os.path.isdir('/content/dgx-spark-hijinks'):
    sh('git clone --depth 50 https://github.com/jethac/dgx-spark-hijinks -b spark/hijinks-022-gemma4-mixed-kv /content/dgx-spark-hijinks')
sh('pip install -q ninja jinja2 packaging')

# Try a real install first (verbose so failures are visible)...
r = sh('pip install -e /content/flashinfer --no-build-isolation', check=False)
FLASHINFER_MODE = 'editable'
if r.returncode != 0:
    print('=== editable install failed (full error above): falling back to source-tree mode ===')
    # Install flashinfer's declared deps, then import the package from the repo.
    # flashinfer declares deps dynamically (setup.py), so install the known set
    sh('pip install -q apache-tvm-ffi filelock requests einops nvidia-ml-py', check=False)
    sys.path.insert(0, '/content/flashinfer')
    os.environ['PYTHONPATH'] = '/content/flashinfer:' + os.environ.get('PYTHONPATH','')
    FLASHINFER_MODE = 'source-tree'

for _attempt in range(4):
    try:
        import flashinfer
        break
    except ModuleNotFoundError as e:
        missing = str(e).split("'")[1]
        pkg = {'tvm_ffi': 'apache-tvm-ffi', 'pynvml': 'nvidia-ml-py'}.get(missing, missing)
        print('missing', missing, '-> pip install', pkg)
        sh(f'pip install -q {pkg}')
else:
    raise RuntimeError('flashinfer import still failing after dep retries')
print('flashinfer', getattr(flashinfer, '__version__', '?'), 'mode:', FLASHINFER_MODE,
      'from', flashinfer.__file__)


In [ ]:
# Cell 4 — probe runner helper.
import json, subprocess, sys, time, pathlib
SCRIPTS = '/content/dgx-spark-hijinks/scripts'
FISRC = '/content/flashinfer'
results = {}

def run_probe(name, script, args, expect):
    out = f'{RESULTS_DIR}/{name}.json'
    cmd = [sys.executable, f'{SCRIPTS}/{script}', *args,
           '--flashinfer-source-root', FISRC, '--output', out]
    t0 = time.time()
    p = subprocess.run(cmd, capture_output=True, text=True, cwd=SCRIPTS)
    wall = time.time() - t0
    rec = {}
    if pathlib.Path(out).exists():
        rec = json.load(open(out))
    ok = rec.get('ok', rec.get('all_ok', False))
    err = (rec.get('error') or '')[:160]
    results[name] = {'ok': ok, 'expect': expect, 'wall_s': round(wall, 1),
                     'record': rec, 'stderr_tail': p.stderr[-400:] if p.returncode else ''}
    verdict = 'PASS' if ok else 'RED'
    match = (verdict == expect)
    print(f"[{name}] {verdict} (expected {expect}) {'== GB10' if match else '!! DIVERGES FROM GB10'}"
          f"  wall={wall:.0f}s")
    if err: print('   error:', err)
    return rec
print('runner ready — first probe pays the JIT (~5-10 min on a fresh cache), rest are fast')

In [ ]:
# Cell 5 — A-series: NVFP4-KV VO-split gates (the Block A equivalents).
VOSPLIT = ['--vo-split', '2', '--head-dim', '512', '--kv-container', 'tuple', '--causal',
           '--v-scale-layout', 'linear', '--no-deswizzle-flag', '--layouts', 'NHD', 'HND',
           '--cosine-threshold', '0.9999']
run_probe('A1_vosplit_batch4', 'flashinfer_nvfp4_kv_probe.py',
          VOSPLIT + ['--batch-size', '4', '--kv-len', '96', '--qo-len', '16', '--page-size', '16'], 'PASS')
run_probe('A2_vosplit_qolen1', 'flashinfer_nvfp4_kv_probe.py',
          VOSPLIT + ['--batch-size', '4', '--kv-len', '96', '--qo-len', '1'], 'PASS')
run_probe('A3_vosplit_signed', 'flashinfer_nvfp4_kv_probe.py',
          VOSPLIT + ['--batch-size', '2', '--kv-len', '64', '--qo-len', '16', '--signed-values'], 'PASS')
run_probe('A4_d128_control', 'flashinfer_nvfp4_kv_probe.py',
          ['--head-dim', '128', '--kv-container', 'tuple', '--causal', '--v-scale-layout', 'linear',
           '--no-deswizzle-flag', '--layouts', 'NHD', 'HND', '--cosine-threshold', '0.9999'], 'PASS')

In [ ]:
# Cell 6 — measured-geometry rows (real Gemma 4 global-layer head counts).
for geom in ('e4b', '31b'):
    run_probe(f'geom_{geom}_bf16_vo256', 'vllm_gemma4_mixed_kv_probes.py',
              ['--probe', 'fa2-vo-split-d512-vo256', '--geometry', geom], 'PASS')

In [ ]:
# Cell 7 — the trait-guard questions: do GB10's REDs reproduce on sm_120?
# fp8 (512,256): GB10 rejects via the 1-byte-dtype term. Same here?
run_probe('fp8_vo256_trait', 'vllm_gemma4_mixed_kv_probes.py',
          ['--probe', 'fa2-vo-split-d512-vo256-fp8kv', '--geometry', 'default'], 'RED')
# bf16 symmetric D=512 (no split): GB10 rejects (the original wall).
run_probe('bf16_d512_symmetric', 'vllm_gemma4_mixed_kv_probes.py',
          ['--probe', 'fa2-bf16-d512'], 'RED')

In [ ]:
# Cell 8 — summary table + verdict.
import torch
cap = torch.cuda.get_device_capability(0)
print(f'=== sm_{cap[0]}{cap[1]} probe matrix vs GB10 (sm_121) reference - notebook {NOTEBOOK_VERSION} ===')
all_match = True
for name, r in results.items():
    verdict = 'PASS' if r['ok'] else 'RED'
    match = verdict == r['expect']
    all_match &= match
    cos = ''
    rec = r['record']
    for k in ('cases', 'results'):
        if isinstance(rec.get(k), list):
            cos = ' | '.join(f"{c.get('layout','?')}:{c.get('cosine', c.get('cosine_similarity','?'))}"
                              for c in rec[k] if isinstance(c, dict))
    print(f"{'OK ' if match else '>>>'} {name:24s} {verdict:4s} (exp {r['expect']:4s}) {cos}")
    if not r['ok'] and r.get('stderr_tail'): print('      ', r['stderr_tail'].strip().splitlines()[-1][:140])
print()
print('VERDICT: family-wide — all rows match GB10' if all_match else
      'VERDICT: DIVERGENCE — rows marked >>> differ from GB10; per-arch finding, save everything')
summary_path = f'{RESULTS_DIR}/SUMMARY_sm{cap[0]}{cap[1]}_{NOTEBOOK_VERSION}.json'
import json as _j
_j.dump({'notebook_version': NOTEBOOK_VERSION, 'rows': {k: {kk: vv for kk, vv in v.items() if kk != 'record'} for k, v in results.items()}},
        open(summary_path, 'w'), indent=2)
print('summary ->', summary_path)

## After the run
1. The JSONs + `SUMMARY_*.json` are in your Drive (or `/content/results_colab_sm120` if no Drive).
2. Hand the directory back to the campaign: drop it into
   `results/colab_sm120_<date>/` on branch `spark/hijinks-022-gemma4-mixed-kv` (Claude
   will commit + fold into the ledger and upstream drafts).
3. **If everything matched GB10**: the upstream issues become two-platform reports.
   **If anything diverged**: that row is a per-architecture finding — arguably even more
   interesting. Either outcome is a win; that is the nice thing about probe matrices.

# Session 2 - vllm#40677 reproduction + fix demo (FOSSA)

BEFORE: stock vLLM rejects FLASHINFER for Gemma 4 NVFP4 on this exact hardware class
(the open issue, verbatim). AFTER: the campaign branch serves Gemma 4 E4B with full
NVFP4 KV on the same GPU. **Prereqs:** cells 1-1c run this session (CUDA 13 env);
HF token for gemma weights (Colab secrets key `HF_TOKEN`). **The build cell (F2) is
~1-2 h of CPU one time** - the wheel caches to Drive and is skipped on re-runs.


In [ ]:
# F0 - HF auth + session prereqs
import os, subprocess, json, time, pathlib
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF token loaded from Colab secrets')
except Exception as e:
    print('No HF_TOKEN secret -', e, '- gemma downloads will fail until you add one')
assert os.environ.get('CUDA_HOME'), 'run cells 1b/1c first (CUDA 13 bootstrap)'
WHEEL_DIR = RESULTS_DIR.replace('results_colab_sm120', 'wheels_sm120a')
os.makedirs(WHEEL_DIR, exist_ok=True)
print('wheel cache:', WHEEL_DIR)


In [ ]:
# F1 - BEFORE: stock vLLM rejection (the #40677 repro). ~10 min incl. model pull.
import subprocess, time, re, json
subprocess.run('python -m venv /content/stock_venv && /content/stock_venv/bin/pip install -q vllm', shell=True)
log = open('/content/before_server.log', 'w')
proc = subprocess.Popen(
    '/content/stock_venv/bin/python -m vllm.entrypoints.openai.api_server --model google/gemma-4-E4B-it'
    ' --attention-backend FLASHINFER --kv-cache-dtype nvfp4 --max-model-len 4096',
    shell=True, stdout=log, stderr=subprocess.STDOUT)
verdict = None
for _ in range(240):
    time.sleep(5)
    txt = open('/content/before_server.log').read()
    m = re.search(r'.*(not valid for this configuration|head_size not supported|No available attention backend).*', txt)
    if m:
        verdict = m.group(0)
        break
    if proc.poll() is not None and 'Traceback' in txt:
        verdict = txt[-1500:]
        break
proc.kill()
subprocess.run('pkill -f api_server', shell=True)
print('BEFORE verdict (the #40677 rejection):')
print(verdict or 'NO REJECTION SEEN - inspect /content/before_server.log')
json.dump({'notebook_version': NOTEBOOK_VERSION, 'before_rejection': verdict},
          open(f'{RESULTS_DIR}/session2_before_FOSSA.json', 'w'), indent=2)


In [ ]:
# F2 - build the campaign vLLM wheel for sm_120a (ONE TIME, ~1-2 h; Drive-cached).
import glob, subprocess, os
cached = glob.glob(f'{WHEEL_DIR}/vllm-*.whl')
if cached:
    print('cached wheel found, skipping build:', cached[0])
else:
    subprocess.run('git clone --depth 200 https://github.com/jethac/vllm -b spark/hijinks-022-gemma4-mixed-kv /content/vllm-022', shell=True)
    # Build deps discovered the hard way on stock Colab (each was a real error):
    subprocess.run('pip install -q --upgrade "setuptools>=77" wheel setuptools-rust cmake setuptools-scm', shell=True)
    if subprocess.run('which cargo', shell=True, capture_output=True).returncode != 0:
        subprocess.run('curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal', shell=True, capture_output=True)
    os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
    assert os.environ.get('CUDA_HOME'), 'CUDA_HOME unset - re-run cell 1c (restarts clear it)'
    env = dict(os.environ, TORCH_CUDA_ARCH_LIST='12.0a', MAX_JOBS='4',
               VLLM_INSTALL_PUNICA_KERNELS='0')
    r = subprocess.run(f'pip wheel --no-build-isolation --no-deps -w {WHEEL_DIR} /content/vllm-022',
                       shell=True, env=env, capture_output=True, text=True)
    print(r.stdout[-2500:])
    print(r.stderr[-4000:])
    cached = glob.glob(f'{WHEEL_DIR}/vllm-*.whl')
    assert cached, 'build failed - paste the tail above back to Claude'
print('wheel:', cached[0])


In [ ]:
# F3 - install the wheel + verify the writer latch (the 3-second provenance gate).
import glob, subprocess, os
wheel = glob.glob(f'{WHEEL_DIR}/vllm-*.whl')[0]
subprocess.run(f'pip install -q {wheel}', shell=True)
env = dict(os.environ, VLLM_NVFP4_KV_LINEAR_V_SF='1')
r = subprocess.run('python /content/dgx-spark-hijinks/scripts/nvfp4_linear_latch_diag.py',
                   shell=True, env=env, capture_output=True, text=True)
print(r.stdout[-1200:] or r.stderr[-1200:])
assert 'LINEAR' in (r.stdout + r.stderr).upper(), 'latch diag did not confirm linear writer - STOP, do not trust AFTER numbers'


In [ ]:
# F4 - AFTER: E4B serves with full NVFP4 KV on this RTX PRO 6000.
import subprocess, time, json, re, urllib.request
env_extra = 'VLLM_NVFP4_KV_VOSPLIT=1 VLLM_NVFP4_KV_LINEAR_V_SF=1'
log = open('/content/after_server.log', 'w')
proc = subprocess.Popen(
    f'{env_extra} python -m vllm.entrypoints.openai.api_server'
    ' --model google/gemma-4-E4B-it --kv-cache-dtype nvfp4 --language-model-only'
    ' --max-model-len 8192 --gpu-memory-utilization 0.80 --port 8000',
    shell=True, stdout=log, stderr=subprocess.STDOUT)
ready = False
for _ in range(240):
    time.sleep(5)
    try:
        urllib.request.urlopen('http://localhost:8000/v1/models', timeout=2)
        ready = True
        break
    except Exception:
        pass
    if proc.poll() is not None:
        break
txt = open('/content/after_server.log').read()
proof = [l for l in txt.splitlines() if re.search(r'VO split|LINEAR_V_SF|KV cache size|FLASHINFER', l)]
print('READY' if ready else 'DID NOT START - tail follows')
for l in proof[-6:]:
    print(l)
result = {'notebook_version': NOTEBOOK_VERSION, 'ready': ready, 'proof_lines': proof}
if ready:
    def chat(prompt, max_tok=128):
        body = json.dumps({'model': 'google/gemma-4-E4B-it', 'temperature': 0,
                           'max_tokens': max_tok, 'messages': [{'role': 'user', 'content': prompt}]}).encode()
        t0 = time.time()
        r = json.load(urllib.request.urlopen(urllib.request.Request(
            'http://localhost:8000/v1/chat/completions', body, {'Content-Type': 'application/json'})))
        return r['choices'][0]['message']['content'], time.time() - t0
    text, _ = chat('In one sentence, what is NVFP4 quantization?')
    print('completion:', text[:300])
    reps = [chat('Count from 1 to 50 in words.', 256) for _ in range(3)]
    result['completion'] = text
    result['decode_tok_s'] = [round(256 / w, 2) for _, w in reps]
    print('decode tok/s (incl prefill):', result['decode_tok_s'])
else:
    result['log_tail'] = txt[-2000:]
    print(txt[-2000:])
proc.kill()
subprocess.run('pkill -f api_server', shell=True)
json.dump(result, open(f'{RESULTS_DIR}/session2_after_FOSSA.json', 'w'), indent=2)
print('saved -> session2_after_FOSSA.json')


## Session 2 wrap
BEFORE rejection + AFTER serving JSONs are in Drive (`session2_*_FOSSA.json`).
Hand both back to the campaign; together with the probe matrix they make the
vllm#40677 answer fully demonstrated on the reporter's hardware class.
Note: the bf16 Triton-retirement demo (vllm#38887 speed story) is deliberately
absent - the warmup dispatcher crash is family-wide and waits on the fix (task #17).
